# torch-asgc integration check: PyTorch, Hugging Face, Lightning

Run this notebook top to bottom in Colab (or any Linux box with a GPU/CPU). It answers one question: **does the public `torch_asgc` library actually train through vanilla PyTorch, the Hugging Face `Trainer`, and PyTorch Lightning?**

- **No dataset downloads.** Every check uses a tiny synthetic tensor dataset.
- **No long training.** Each integration runs 5-6 optimizer steps.
- **Every check prints PASS or FAIL with a full traceback.** Paste the failing cell's output if something breaks.
- The notebook installs `transformers`, `accelerate`, `datasets`, and `lightning`, then imports the library directly from the repository's `src/` directory (this avoids depending on a wheel build).

The integration checks are:

1. Vanilla PyTorch: `DistributedModel` + `DistributedOptimizer` over `LocalTransportHub`.
2. Real TCP transport: one push/fetch over a localhost socket.
3. Hugging Face: `ASGCTrainer` + `ASGCTrainerCallback` on a 2-layer GPT-2 with random token ids.
4. PyTorch Lightning: `ASGCStrategy` component path and a full `pl.Trainer.fit`.


## 1. Setup

Edit `REPO_URL` / `ASGC_REPO_DIR` only if the repository is somewhere else (for example, a Colab upload). If the notebook is already running inside the repo, it uses that checkout.


In [ ]:
import os
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/ksacheth/parallel-computing.git"
ENV_REPO_DIR = os.environ.get("ASGC_REPO_DIR", "")

candidates = [pathlib.Path(ENV_REPO_DIR)] if ENV_REPO_DIR else []
candidates += [pathlib.Path.cwd(), pathlib.Path("/content/parallel-computing")]
REPO_DIR = next((p for p in candidates if (p / "src" / "torch_asgc").is_dir()), None)

if REPO_DIR is None:
    REPO_DIR = pathlib.Path("/content/parallel-computing")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

print("Repository:", REPO_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers>=4.40", "accelerate>=1.1", "datasets", "lightning>=2.0"],
    check=True,
)
sys.path.insert(0, str(REPO_DIR / "src"))
print("src on sys.path:", sys.path[0])


In [ ]:
import importlib.metadata
import platform
import threading
import time
import traceback

import torch
import torch.nn as nn

import torch_asgc
from torch_asgc import (
    DistributedModel,
    DistributedOptimizer,
    LocalTransportHub,
    ParameterServer,
    StalenessConfig,
    connect,
)
from torch_asgc.plugins import ASGCStrategy, ASGCTrainer, ASGCTrainerCallback

print("Python:", platform.python_version())
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("torch_asgc:", torch_asgc.__version__, "from", torch_asgc.__file__)
for package in ("transformers", "accelerate", "datasets", "lightning", "pytorch-lightning"):
    try:
        print(f"{package}: {importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package}: not installed")

RESULTS = []


def check(name, fn):
    print(f"\n=== {name} ===")
    try:
        detail = fn() or ""
        RESULTS.append({"check": name, "status": "PASS", "detail": detail})
        print("PASS", detail)
    except Exception as exc:
        RESULTS.append({"check": name, "status": "FAIL", "detail": f"{type(exc).__name__}: {exc}"})
        print("FAIL", type(exc).__name__, exc)
        traceback.print_exc()


## 2. Shared helpers

`ParameterServer` owns the authoritative model and runs `serve_forever()` in a daemon thread. Workers talk to it through either `LocalTransportHub` (in-process queues) or a real TCP socket.


In [ ]:
def start_server(model, hub=None, bind_address=None, s_max=10):
    optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
    server = ParameterServer(
        model=model,
        optimizer=optimizer,
        transport=hub,
        bind_address=bind_address,
        staleness_config=StalenessConfig(s_low=2, s_max=s_max, beta=0.5),
        compression_mode="topk",
        initial_strength=0.5,
    )
    thread = threading.Thread(target=server.serve_forever, daemon=True)
    thread.start()
    return server


def hub_server_and_client(worker_id="check_worker"):
    server_model = nn.Linear(8, 2)
    hub = LocalTransportHub()
    server = start_server(server_model, hub=hub)
    client = connect(hub, worker_id=worker_id)
    return server, client, server_model


def safe_close(obj):
    try:
        obj.close()
    except Exception:
        pass


def param_gap(worker, server_model):
    return max(
        (worker.state_dict()[name] - tensor).abs().max().item()
        for name, tensor in server_model.state_dict().items()
    )


## 3. Check 1 - vanilla PyTorch

This is the core public API: wrap an `nn.Module`, wrap any optimizer, and call `step()` in a normal training loop.


In [ ]:
def check_vanilla_pytorch():
    server, client, server_model = hub_server_and_client("vanilla")
    try:
        worker = nn.Linear(8, 2)
        model = DistributedModel(worker, client=client)
        optimizer = DistributedOptimizer(
            base_optimizer=torch.optim.SGD(worker.parameters(), lr=0.05),
            model=model,
            client=client,
            compression_mode="topk",
            rho=0.5,
            error_feedback=True,
            local_steps=1,
        )
        for _ in range(6):
            optimizer.zero_grad()
            x = torch.randn(4, 8)
            y = torch.randn(4, 2)
            loss = ((model(x) - y) ** 2).mean()
            loss.backward()
            optimizer.step(loss_val=float(loss.detach()))

        assert server.version == 6, f"expected 6 applied versions, got {server.version}"
        gap = param_gap(worker, server_model)
        assert gap < 1e-6, f"worker/server parameters diverged by {gap:.2e}"
        return f"6 steps, server version={server.version}, max worker/server param gap={gap:.1e}"
    finally:
        safe_close(client)
        safe_close(server)


check("Vanilla PyTorch (DistributedModel + DistributedOptimizer)", check_vanilla_pytorch)


## 4. Check 2 - real TCP transport

The tests and the checks above use an in-process hub. This check binds a real localhost socket and sends one compressed push and one parameter fetch over it.


In [ ]:
def check_tcp_transport():
    import socket

    probe = socket.socket()
    probe.bind(("127.0.0.1", 0))
    port = probe.getsockname()[1]
    probe.close()

    server_model = nn.Linear(4, 1)
    server = start_server(server_model, bind_address=("127.0.0.1", port))
    client = None
    try:
        time.sleep(0.2)
        client = connect(f"tcp://127.0.0.1:{port}", worker_id="tcp_worker")
        worker = nn.Linear(4, 1)
        model = DistributedModel(worker, client=client)
        optimizer = DistributedOptimizer(
            base_optimizer=torch.optim.SGD(worker.parameters(), lr=0.05),
            model=model,
            client=client,
            compression_mode="topk",
            rho=0.5,
        )
        optimizer.zero_grad()
        x = torch.randn(2, 4)
        loss = model(x).sum()
        loss.backward()
        optimizer.step(loss_val=float(loss.detach()))
        assert server.version == 1, f"expected server version 1, got {server.version}"
        return f"TCP push/fetch on port {port}, server version={server.version}"
    finally:
        if client is not None:
            safe_close(client)
        safe_close(server)


check("TCP transport (real localhost socket)", check_tcp_transport)


## 5. Check 3 - Hugging Face `Trainer`

A 2-layer GPT-2 (32-d embeddings, 64-token vocabulary) trains on random token ids. This exercises `ASGCTrainer.create_optimizer`, the `ASGCTrainerCallback` telemetry hook, and the full `Trainer.train()` loop. Nothing is downloaded.


In [ ]:
def check_huggingface():
    import tempfile

    import numpy as np
    from datasets import Dataset
    from transformers import GPT2Config, GPT2LMHeadModel, TrainingArguments

    server, client, _ = hub_server_and_client("hf")
    try:
        config = GPT2Config(vocab_size=64, n_positions=16, n_embd=32, n_layer=2, n_head=2)
        model = GPT2LMHeadModel(config)

        rng = np.random.default_rng(0)
        token_rows = [rng.integers(0, 64, size=16).tolist() for _ in range(32)]
        dataset = Dataset.from_dict({"input_ids": token_rows})

        def collate(batch):
            input_ids = torch.tensor([row["input_ids"] for row in batch], dtype=torch.long)
            return {
                "input_ids": input_ids,
                "attention_mask": torch.ones_like(input_ids),
                "labels": input_ids.clone(),
            }

        args = TrainingArguments(
            output_dir=tempfile.mkdtemp(prefix="asgc_hf_"),
            max_steps=5,
            per_device_train_batch_size=2,
            logging_steps=1,
            save_strategy="no",
            report_to="none",
            remove_unused_columns=False,
            dataloader_num_workers=0,
        )
        trainer = ASGCTrainer(
            client=client,
            model=model,
            args=args,
            train_dataset=dataset,
            data_collator=collate,
            compression_mode="topk",
            rho=0.5,
        )
        optimizer = trainer.create_optimizer()
        assert isinstance(optimizer, DistributedOptimizer), type(optimizer).__name__
        assert any(isinstance(cb, ASGCTrainerCallback) for cb in trainer.callbacks)

        trainer.train()
        assert server.version >= 5, f"expected at least 5 applied versions, got {server.version}"
        callback_loss = trainer._asgc_callback.last_loss
        assert callback_loss is not None, "ASGCTrainerCallback did not capture a loss"
        return (
            f"HF Trainer trained 5 steps, server version={server.version}, "
            f"last push loss={callback_loss:.4f}"
        )
    finally:
        safe_close(client)
        safe_close(server)


check("Hugging Face Trainer (ASGCTrainer + callback)", check_huggingface)


## 6. Check 4 - PyTorch Lightning

Two checks, because they fail for different reasons:

- **Component path:** `ASGCStrategy.setup_model` and `setup_optimizers` wrap the module and optimizer, and `optimizer_step` pushes to the server. This is the path the unit tests cover.
- **Full `Trainer.fit`:** Lightning owns the loop and calls the strategy hooks itself. This is the integration that a user actually runs.


In [ ]:
def check_lightning_components():
    from types import SimpleNamespace

    server, client, _ = hub_server_and_client("lightning_components")
    try:
        worker = nn.Linear(4, 1)
        strategy = ASGCStrategy(client=client, compression_mode="topk", rho=0.5)
        wrapped = strategy.setup_model(worker)
        assert isinstance(wrapped, DistributedModel), type(wrapped).__name__

        base_optimizer = torch.optim.SGD(wrapped.parameters(), lr=0.1)
        fake_trainer = SimpleNamespace(
            lightning_module=wrapped,
            optimizers=[base_optimizer],
            callback_metrics={"train_loss": torch.tensor(0.5)},
        )
        optimizers = strategy.setup_optimizers(fake_trainer)
        assert isinstance(optimizers[0], DistributedOptimizer), type(optimizers[0]).__name__

        x = torch.randn(2, 4)
        loss = wrapped(x).sum()
        loss.backward()
        strategy.optimizer_step(optimizers[0])
        time.sleep(0.1)
        assert server.version > 0, "strategy.optimizer_step did not reach the server"
        return f"strategy components pushed version {server.version}"
    finally:
        safe_close(client)
        safe_close(server)


check("Lightning strategy components (setup + optimizer_step)", check_lightning_components)


In [ ]:
def check_lightning_trainer():
    import lightning.pytorch as pl

    class LitModel(pl.LightningModule):
        def __init__(self):
            super().__init__()
            self.net = nn.Linear(8, 2)

        def training_step(self, batch, batch_idx):
            x, y = batch
            return nn.functional.cross_entropy(self.net(x), y)

        def configure_optimizers(self):
            return torch.optim.SGD(self.parameters(), lr=0.05)

    class SyntheticData(torch.utils.data.Dataset):
        def __len__(self):
            return 64

        def __getitem__(self, index):
            generator = torch.Generator().manual_seed(index)
            x = torch.randn(8, generator=generator)
            y = torch.randint(0, 2, (1,), generator=generator)
            return x, y

    server, client, _ = hub_server_and_client("lightning_fit")
    try:
        loader = torch.utils.data.DataLoader(SyntheticData(), batch_size=4)
        strategy = ASGCStrategy(client=client, compression_mode="topk", rho=0.5, device="cpu")
        trainer = pl.Trainer(
            max_steps=5,
            accelerator="cpu",
            devices=1,
            logger=False,
            enable_checkpointing=False,
            enable_progress_bar=False,
            enable_model_summary=False,
            strategy=strategy,
        )
        trainer.fit(LitModel(), train_dataloaders=loader)
        assert server.version > 0, "Trainer.fit did not push any update"
        return f"Trainer.fit completed, server version={server.version}"
    finally:
        safe_close(client)
        safe_close(server)


check("Lightning Trainer.fit (full loop)", check_lightning_trainer)


## 7. Summary

Every check is independent: a failure in the Lightning full-loop check does not invalidate the strategy components or the Hugging Face path, and vice versa.


In [ ]:
import pandas as pd

summary = pd.DataFrame(RESULTS)
print(summary.to_string(index=False))
summary.style.apply(
    lambda row: ["background-color: #d7f5d7" if row["status"] == "PASS" else "background-color: #f8d7da"] * len(row),
    axis=1,
)


## How to read the result

- **All PASS:** the library's vanilla PyTorch, TCP, Hugging Face, and Lightning integrations all work in this environment. The checks use tiny models and synthetic data, so this validates the integration surface, not compression quality.
- **Vanilla/TCP PASS, a framework check FAILS:** the core library is fine. Paste the traceback; framework adapter code in `src/torch_asgc/plugins/` needs updating for that framework version.
- **Lightning components PASS, Trainer.fit FAILS:** the wrapper hooks are correct, but the installed Lightning version's strategy lifecycle differs. Report the traceback and the printed `lightning` version.
- **HF FAILS at `accelerate` import:** run `pip install -U accelerate` and restart the runtime. This is an environment issue, not an ASGC issue.

### What is intentionally not tested here

- Multi-process or multi-machine deployment (this notebook uses daemon threads and one localhost socket).
- Compression quality, staleness behavior, and wall-clock gains. Those are in `transformer_asgc_cuda.ipynb`, `fmnist_asgc.ipynb`, and `cifar_asgc.ipynb`.
- Persistence/resume of server state across restarts.
